# Ch.1　データを「読む」── 【講師用完全版】

| 項目 | 内容 |
|------|------|
| **使うデータ** | ワインデータ 178件・13特徴量・3品種 |
| **作るもの** | 品種別・主要指標の集計表 |
| **実務での対応物** | 月次売上レポート・KPI ダッシュボードの集計表 |
| **演習時間** | 40分（問3まで完了で合格） |

> 📌 **講師メモ：** 「コードを覚えること」より「4点確認の習慣をつけること」が目的。  
> `head → info → describe → isnull` の順番を繰り返し言葉にする。

---
## 🤖 Copilot の使い方

**URL：** https://copilot.microsoft.com

**質問の型（5点を揃える）：**
```
【やりたいこと】〇〇したい
【使うライブラリ】pandas
【データの形】df という DataFrame、178件。「品種名」「alcohol」などの列がある
【環境】Python 3.8.6、Windows、JupyterLab
【困っていること】〇〇の書き方がわからない
```

---
## 準備：ライブラリとデータを読み込む

> ⚠️ **まず最初にこのセルを実行してください。**

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine

wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種名"] = [wine.target_names[t] for t in wine.target]

print(f"✅ 読み込み完了　行数: {len(df)}  列数: {len(df.columns)}")

**✅ 「行数: 178  列数: 14」と表示されれば OK です。**

---
## STEP 1　先頭を見て「データの形」を掴む

### ❓ なぜ最初に先頭を見るのか

> **地図を広げるように、まず「何が入っているか」を確認します。**  
> 新しいデータをもらったとき、最初の5行を見るだけで  
> 「どんな列があるか・どんな値が入っているか」が把握できます。

---
### 📝 タスク 1-1　先頭5行を表示してください

**何をするか：** `df.head()` を実行する（それだけでOK）

In [ ]:
# 先頭5行を表示する
df.head()

**見るポイント：**
- どんな列（変数）がありますか？
- 数値の大きさに差はありますか？（`proline` は他より桁が大きい）

---
### 📝 タスク 1-2　末尾5行を表示してください

**何をするか：** `head()` の代わりになる関数を使う  
**Copilot に聞く場合：** 「pandas で DataFrame の末尾5行を表示するメソッドを教えて」

In [ ]:
# 末尾5行を表示する
df.tail()

---
## STEP 2　型と欠損を確認する

### ❓ なぜ型と欠損を確認するのか

> **型が間違っていたり欠損があると、計算でエラーが起きます。**  
> 「alcohol が object 型になっている」→ 平均を計算しようとするとエラー。  
> 実務では必ずこの確認を行います。

---
### 📝 タスク 2-1　列の型・件数・欠損を確認してください

**何をするか：** `df.info()` を実行する  
**確認するポイント：** `Non-Null Count` が全列 178 になっているか

In [ ]:
# 型・件数・欠損値を確認する
df.info()

**見るポイント：**
- `Non-Null Count` がすべて 178 → 欠損なし ✅
- `Dtype` がすべて `float64` → 数値として正しく読めている ✅

---
## STEP 3　基本統計量でばらつきを確認する

### ❓ なぜ統計量を確認するのか

> **「平均」だけでは見えない「ばらつき」を確認します。**  
> `std`（標準偏差）が大きい列 → 品種間の差が出やすい → 分類に効く変数の候補。

---
### 📝 タスク 3-1　全列の基本統計量を表示してください

**何をするか：** `df.describe()` を実行する（`.round(2)` で小数2桁に丸める）

In [ ]:
# 平均・標準偏差・最大最小を確認する
df.describe().round(2)

**見るポイント：**
- `std` が大きい列 → 値のばらつきが大きい → 品種を区別しやすい変数の候補
- `mean` と `max` の差が極端に大きい列 → 外れ値の可能性

---
### 📝 タスク 3-2　alcohol 列だけの統計量を確認してください

**何をするか：** 列を1つだけ指定して `describe()` を実行する  
**Copilot に聞く場合：** 「pandas で特定の列だけ describe() する書き方を教えて」

In [ ]:
# alcohol 列だけの統計量を確認する
df["alcohol"].describe().round(2)

---
## STEP 4　欠損値を数える

### ❓ なぜ欠損値を確認するのか

> **欠損値（NaN）は「見えない地雷」です。**  
> 1つあるだけで平均や合計の計算結果がおかしくなります。  
> 実務ではほぼ必ず欠損がありますが、今日はゼロのデータで始めます。

---
### 📝 タスク 4-1　列ごとの欠損値の件数を確認してください

**何をするか：** `df.isnull().sum()` を実行する

In [ ]:
# 欠損値の件数を確認する
df.isnull().sum()

**見るポイント：**
- すべて **0** → 欠損なし → そのまま分析に進める ✅

---
### 📝 タスク 4-2　欠損が1件以上ある列だけを表示してください

**何をするか：** `isnull().sum()` の結果から条件でフィルタする  
**Copilot に聞く場合：** 「pandas の Series で値が 0 より大きい要素だけ取り出す方法を教えて」

In [ ]:
# 欠損が1件以上ある列だけ表示する
missing = df.isnull().sum()
missing[missing > 0]

---
### 📌 補足｜欠損値があった場合の対処法（読むだけでOK）

> 今日は欠損なしのデータですが、実務では欠損が必ずあります。

| 状況 | 対処 |
|------|------|
| 欠損が全体の5%以下 | `df.dropna()` で欠損行を削除 |
| 特定の列の欠損が多い | `df.drop(columns=["列名"])` でその列ごと削除 |
| 重要な列の欠損 | `df["列名"].fillna(df["列名"].mean())` で平均値補完 |

In [ ]:
# 補足コード：欠損値への対処パターン（読むだけでOK）
# ① 欠損のある行をすべて削除する
# df.dropna()

# ② 特定の列の欠損を平均値で補完する
# df["列名"] = df["列名"].fillna(df["列名"].mean())

# ③ 欠損がある行だけ確認する
# df[df["列名"].isnull()]

---
## 問1　品種ごとのデータ件数を確認する ★☆☆

### ❓ なぜ件数を確認するのか

> **データが特定のグループに偏っていると、分析結果がゆがみます。**  
> 例）3品種が「100件・50件・5件」なら、5件の品種は分析が不安定になります。  
> まず「どのグループが何件あるか」を確認するのが第一歩です。

---
### 📝 タスク 1-1　品種名ごとのデータ件数を確認してください

**何をするか：** `value_counts()` で品種ごとの件数を数える

In [ ]:
# 品種名ごとの件数を確認する
df["品種名"].value_counts()

---
### 📝 タスク 1-2　件数を割合（%）でも確認してください

**何をするか：** `value_counts()` に引数を1つ追加するだけ  
**Copilot に聞く場合：** 「pandas の value_counts で件数でなく割合を表示する方法を教えて」

In [ ]:
# 件数を割合（0〜1）で確認する
df["品種名"].value_counts(normalize=True).round(2)

**✅ チェックポイント**
- [ ] 3品種の件数が表示されているか（合計 178件）
- [ ] 割合の合計は 1.0（100%）になっているか
- [ ] 最も多い品種と少ない品種の差はどれくらいか

---
## 問2　品種別の集計表を作る ★★☆

### ❓ なぜ groupby を使うのか

> **「全体の平均」より「グループごとの平均」の方が、差が見えます。**  
> Excel のピボットテーブルと同じ考え方です。  
> 「仕分けてから計算する」が `groupby` の本質です。

---
### 📝 タスク 2-1　品種ごとの alcohol 平均を計算してください

**何をするか：** `df.groupby("品種名")["alcohol"].mean()` を実行する

In [ ]:
# 品種ごとの alcohol の平均を計算する
df.groupby("品種名")["alcohol"].mean().round(2)

---
### 📝 タスク 2-2　品種ごとの proline 平均を確認してください

**何をするか：** タスク 2-1 の列名を `"proline"` に変えるだけ

In [ ]:
# 品種ごとの proline の平均を計算する
df.groupby("品種名")["proline"].mean().round(2)

---
### 📝 タスク 2-3　alcohol と proline を1つの表にまとめてください

**何をするか：** `["alcohol"]` を `[["alcohol", "proline"]]` に変える（角括弧が2重）

In [ ]:
# alcohol と proline を1つの表にまとめる
df.groupby("品種名")[["alcohol", "proline"]].mean().round(2)

---
### 📝 タスク 2-4　平均・最大・最小・標準偏差をまとめて出してください

**何をするか：** `.mean()` の代わりに `.agg([...])` を使う  
**Copilot に聞く場合：** 「pandas の groupby で mean・max・min・std を一度に出す方法を教えて」

In [ ]:
# 平均・最大・最小・標準偏差をまとめて出す
df.groupby("品種名")[["alcohol", "proline"]].agg(["mean", "max", "min", "std"]).round(2)

**✅ チェックポイント**
- [ ] 3品種の数値が1つの表にまとまっているか
- [ ] 品種間で差が大きい変数と小さい変数を1つずつ挙げられるか
- [ ] `std` が大きい変数はどれか（ばらつきが大きい = 品種を区別しやすい候補）

---
## 問3　集計結果から考察する ★★☆（最重要・AI 禁止）

> ⛔ **この問は自分の頭で考えてください。AI は使わないこと。**

> 📌 **なぜ大事なのか：**  
> 数字を見て「何が言えるか」を言葉にすることが、  
> データサイエンティストの最も重要なスキルです。  
> Ch.2 で立てた仮説を Ch.3 の特徴量重要度で「答え合わせ」します。

---

### 良い考察の書き方

```
① 事実（数字で）：「〇〇品種の △△ 平均は他品種より□□ 大きかった」
② 解釈（自分の考え）：「→ △△ は品種を区別しやすい変数だと思う」
③ 予測（Ch.3 への接続）：「→ Ch.3 では △△ の重要度が高くなると予測する」
```

---

### 考察を書いてください

**【発見 1】数字で気づいたこと（具体的な数字を入れて書く）**

>

**【解釈】それはどういう意味だと思うか**

>

**【予測】Ch.3 の分類モデルで重要になりそうな変数はどれか（理由も書く）**

>

**✅ 振り返りチェック**
- [ ] 「〇倍」「〇ポイント差」など **具体的な数字** を使えたか
- [ ] 自分なりの **解釈（なぜそう思うか）** を書けたか
- [ ] Ch.3 で答え合わせする **予測** が書けたか

---
## 問4（発展）　条件でデータを絞り込む ★★★

> ⏰ **時間が余った人だけ取り組んでください**

### ❓ なぜ絞り込みを使うのか

> **「特定の条件を満たすデータだけ」を分析したいときに使います。**  
> 例）「売上 100万円以上の顧客だけ」「年齢 30代だけ」など。

**絞り込みの書き方：**
```
1条件：df[df["列名"] >= 値]
AND条件：df[(条件1) & (条件2)]    ← 括弧が必須！
```

---
### 📝 タスク 4-1　アルコール度数が 13.5 以上のワインを取り出してください

In [ ]:
# alcohol >= 13.5 のワインを取り出す
high_alc = df[df["alcohol"] >= 13.5]
print(f"{len(high_alc)} 件")

---
### 📝 タスク 4-2　alcohol ≥ 13.5 かつ proline ≥ 700 の AND 条件で絞り込んでください

**注意：** AND 条件では各条件を `()` で囲む必要があります

In [ ]:
# AND 条件で絞り込む（括弧が必須）
premium = df[(df["alcohol"] >= 13.5) & (df["proline"] >= 700)]
print(f"{len(premium)} 件")

---
### 📝 タスク 4-3　絞り込んだデータで品種ごとの件数と平均を確認してください

In [ ]:
# 絞り込み後の品種ごとの件数を確認する
premium["品種名"].value_counts()

In [ ]:
# 絞り込み後の品種ごとの平均を確認する
premium.groupby("品種名")[["alcohol", "proline"]].mean().round(2)

**✅ チェックポイント**
- [ ] 絞り込み後は全体（178件）より件数が少ないか
- [ ] AND 条件でさらに件数が絞られているか
- [ ] 絞り込み後に最も多く残った品種はどれか
- [ ] 問2・問3 の考察と一致しているか